# 05 Q2: รูปทรงของ peak hour เปลี่ยนไปไหม

Pain point ของ BART: ตารางเดินรถออกแบบไว้สำหรับ peak ก่อน COVID ถ้า peak เปลี่ยนรูป จะมีรถเกินในบางชั่วโมงและแน่นในบางชั่วโมง

ข้อมูลที่ใช้
- system_hour และ station_hour เฉพาะวันทำงาน normal
- เทียบ ม.ค.-เม.ย. 2019 กับ ม.ค.-เม.ย. 2023

EDA ดูรูปทรงแบบสัดส่วนไปแล้ว (peak เช้าเล็กลง กลางวันแน่นขึ้น) notebook นี้เจาะว่าชั่วโมงไหนฟื้นเท่าไร peak เลื่อนไปไหน และต่างกันอย่างไรในแต่ละสถานี

ข้อจำกัด: ไม่มีข้อมูลตารางเดินรถและความจุขบวน จึงบอกได้แค่ว่าความต้องการเปลี่ยนไปอย่างไร ไม่ได้บอกว่าตอนนี้รถเกินหรือขาดกี่ขบวน

In [1]:
import sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()
sys.path.append(str(ROOT / "src"))

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from bart import viz
from bart.config import create_views, duck, fetch_value
from bart.perf import stage
from bart.periods import AFTER, BEFORE, PERIODS, period_case
from bart.stations import STATIONS

viz.notebook_mode()

con = duck()
create_views(con)

## 1. ผู้โดยสารรายชั่วโมง: ค่าจริงและ recovery %

In [2]:
hourly_query = f"""
    CREATE OR REPLACE TEMP TABLE hourly AS
    WITH hours_in_period AS (
        SELECT
            {period_case("s.date")} AS period,
            s.hour,
            s.trips
        FROM system_hour AS s
        JOIN calendar AS c ON c.date = s.date
        WHERE c.day_status = 'normal' AND c.is_workday
    )
    SELECT period, hour, avg(trips) AS trips
    FROM hours_in_period
    WHERE period IS NOT NULL
    GROUP BY period, hour
"""

with stage("05.hourly"):
    con.execute(hourly_query)

by_hour = con.sql(f"""
    SELECT
        b.hour,
        round(b.trips)                       AS trips_2019,
        round(a.trips)                       AS trips_2023,
        round(b.trips - a.trips)             AS lost_per_day,
        round(100 * a.trips / b.trips, 1)    AS recovery_pct
    FROM hourly AS b
    JOIN hourly AS a ON a.hour = b.hour AND a.period = '{AFTER}'
    WHERE b.period = '{BEFORE}'
    ORDER BY b.hour
""").df()

by_hour

[perf] 05.hourly: 0.01s | 0.86 cores | peak RAM 0.232 GB


,hour,trips_2019,trips_2023,lost_per_day,recovery_pct
0,0,1585.0,691.0,894.0,43.6
1,1,307.0,160.0,146.0,52.3
2,2,1.0,17.0,-15.0,1146.0
3,3,2.0,2.0,1.0,74.6
4,4,221.0,43.0,179.0,19.2
5,5,4537.0,1812.0,2725.0,39.9
6,6,15030.0,5904.0,9125.0,39.3
7,7,33264.0,11985.0,21279.0,36.0
8,8,50760.0,16908.0,33852.0,33.3
9,9,38971.0,11716.0,27255.0,30.1


In [3]:
line_2019 = go.Scatter(
    x=by_hour["hour"],
    y=by_hour["trips_2019"],
    mode="lines+markers",
    name=BEFORE,
    line=dict(color=viz.CONTEXT, width=3),
    hovertemplate="%{x}:00 - %{y:,.0f} trips<extra>%{fullData.name}</extra>",
)
line_2023 = go.Scatter(
    x=by_hour["hour"],
    y=by_hour["trips_2023"],
    mode="lines+markers",
    name=AFTER,
    line=dict(color=viz.ACCENT, width=3),
    hovertemplate="%{x}:00 - %{y:,.0f} trips<extra>%{fullData.name}</extra>",
)

fig = go.Figure([line_2019, line_2023])
fig.update_xaxes(title_text="hour", dtick=2)
fig.update_yaxes(title_text="average trips per workday hour", tickformat="~s")

peak_hour_2019 = 0
peak_trips_2019 = 0.0
peak_trips_2023 = 0.0
for row in by_hour.to_dict("records"):
    if row["trips_2019"] > peak_trips_2019:
        peak_hour_2019 = int(row["hour"])
        peak_trips_2019 = row["trips_2019"]
    if row["trips_2023"] > peak_trips_2023:
        peak_trips_2023 = row["trips_2023"]

title = (
    f"The busiest hour fell from {peak_trips_2019 / 1000:.0f}k trips ({peak_hour_2019}:00, 2019) "
    f"to {peak_trips_2023 / 1000:.0f}k in 2023"
)
subtitle = "Average trips per hour on normal workdays, Jan-Apr 2019 vs Jan-Apr 2023"
viz.titled(fig, title, subtitle)
viz.save(fig, "05_hourly_levels")
fig

recovery % ของแต่ละชั่วโมง ไม่แสดงช่วงตี 1-4 เพราะแทบไม่มีผู้โดยสาร (สัดส่วนจะแกว่งมาก)

In [4]:
service_rows = []
for row in by_hour.to_dict("records"):
    if row["hour"] >= 5 or row["hour"] == 0:
        service_rows.append(row)
service_hours = pd.DataFrame(service_rows)

colors = []
for hour in service_hours["hour"]:
    if 6 <= hour <= 9 or 15 <= hour <= 18:
        colors.append(viz.CONTRAST)
    else:
        colors.append(viz.ACCENT)

labels = []
for hour in service_hours["hour"]:
    labels.append(f"{int(hour)}:00")

bars = go.Bar(
    x=labels,
    y=service_hours["recovery_pct"],
    marker_color=colors,
    text=service_hours["recovery_pct"].round(0).astype(int).astype(str) + "%",
    textposition="outside",
    customdata=service_hours[["trips_2019", "trips_2023"]],
    hovertemplate="%{x}: %{customdata[1]:,.0f} vs %{customdata[0]:,.0f} trips<extra></extra>",
)

fig = go.Figure(bars)
fig.update_layout(hovermode="closest", showlegend=False)
fig.update_yaxes(title_text="% of Jan-Apr 2019", ticksuffix="%", range=[0, 75])

title = "Midday recovered most (46-50%); the 9:00 and 18:00 commute hours least (~30%)"
subtitle = "Trips per workday hour in Jan-Apr 2023 as % of Jan-Apr 2019. Orange = AM peak (6-9) and PM peak (15-18)"
viz.titled(fig, title, subtitle)
viz.save(fig, "05_recovery_by_hour")
fig

## 2. ตัวชี้วัดรูปทรง peak

- busiest hour: ชั่วโมงที่คนเยอะที่สุด และสัดส่วนของวัน
- peak-to-average: ชั่วโมงที่คนเยอะที่สุด หารด้วยค่าเฉลี่ยของ 24 ชั่วโมง
- AM / PM share: สัดส่วนของวันในช่วง 6:00-9:59 และ 15:00-18:59
- hours for 50%: ต้องใช้กี่ชั่วโมงที่คนเยอะที่สุดรวมกันถึงจะได้ครึ่งหนึ่งของทริปทั้งวัน ถ้ามากขึ้นแปลว่า peak แบนลง

In [5]:
def shape_metrics(trips_by_hour):
    total = 0.0
    for trips in trips_by_hour.values():
        total += trips

    busiest_hour = None
    busiest_trips = -1.0
    am_trips = 0.0
    pm_trips = 0.0
    for hour, trips in trips_by_hour.items():
        if trips > busiest_trips:
            busiest_hour = hour
            busiest_trips = trips
        if 6 <= hour <= 9:
            am_trips += trips
        if 15 <= hour <= 18:
            pm_trips += trips

    hours_for_half = 0
    covered = 0.0
    for trips in sorted(trips_by_hour.values(), reverse=True):
        hours_for_half += 1
        covered += trips
        if covered >= total / 2:
            break

    return {
        "busiest_hour": busiest_hour,
        "busiest_hour_pct": round(100 * busiest_trips / total, 1),
        "peak_to_average": round(busiest_trips / (total / 24), 2),
        "am_share_pct": round(100 * am_trips / total, 1),
        "pm_share_pct": round(100 * pm_trips / total, 1),
        "hours_for_50pct": hours_for_half,
    }


hourly_rows = con.sql("SELECT period, hour, trips FROM hourly").fetchall()

trips_by_period = {}
for period, hour, trips in hourly_rows:
    if period not in trips_by_period:
        trips_by_period[period] = {}
    trips_by_period[period][hour] = trips

metric_rows = []
for period in PERIODS:
    metrics = shape_metrics(trips_by_period[period])
    metrics["period"] = period
    metric_rows.append(metrics)

shape = pd.DataFrame(metric_rows).set_index("period")
shape

,busiest_hour,busiest_hour_pct,peak_to_average,am_share_pct,pm_share_pct,hours_for_50pct
period,,,,,,
Jan-Apr 2019,8,12.4,2.97,33.7,35.1,5
Jan-Apr 2023,17,11.8,2.83,30.6,35.5,6


## 3. peak เลื่อนเวลาไหม

จุดศูนย์กลางของ peak = เวลาเฉลี่ยถ่วงน้ำหนักด้วยจำนวนทริป
- peak เช้าใช้ชั่วโมง 5-10 ส่วน peak เย็นใช้ชั่วโมง 15-19
- ชั่วโมง 8 หมายถึง 8:00-8:59 จึงบวก 0.5 ให้เป็นกลางชั่วโมง

ดูเป็นรายเดือน วันทำงาน normal ตั้งแต่ ม.ค. 2019 ถึง เม.ย. 2023

In [6]:
centre_query = """
    WITH workday_hours AS (
        SELECT
            date_trunc('month', s.date)::DATE AS month,
            s.hour,
            s.trips
        FROM system_hour AS s
        JOIN calendar AS c ON c.date = s.date
        WHERE c.day_status = 'normal'
          AND c.is_workday
          AND s.date BETWEEN DATE '2019-01-01' AND DATE '2023-04-30'
    )
    SELECT
        month,
        sum((hour + 0.5) * trips) FILTER (WHERE hour BETWEEN 5 AND 10)
            / sum(trips) FILTER (WHERE hour BETWEEN 5 AND 10)  AS am_centre,
        sum((hour + 0.5) * trips) FILTER (WHERE hour BETWEEN 15 AND 19)
            / sum(trips) FILTER (WHERE hour BETWEEN 15 AND 19) AS pm_centre
    FROM workday_hours
    GROUP BY month
    ORDER BY month
"""

with stage("05.peak_centre"):
    centres = con.sql(centre_query).df()


def clock(hours):
    whole = int(hours)
    minutes = round((hours - whole) * 60)
    if minutes == 60:
        whole += 1
        minutes = 0
    return f"{whole}:{minutes:02d}"


centre_rows = []
for label, (start, end) in PERIODS.items():
    am_values = []
    pm_values = []
    for row in centres.to_dict("records"):
        if pd.Timestamp(start) <= pd.Timestamp(row["month"]) <= pd.Timestamp(end):
            am_values.append(row["am_centre"])
            pm_values.append(row["pm_centre"])
    am = sum(am_values) / len(am_values)
    pm = sum(pm_values) / len(pm_values)
    centre_rows.append({"period": label, "am_peak_centre": clock(am), "pm_peak_centre": clock(pm)})

pd.DataFrame(centre_rows).set_index("period")

[perf] 05.peak_centre: 0.01s | 3.39 cores | peak RAM 0.255 GB


,am_peak_centre,pm_peak_centre
period,,
Jan-Apr 2019,8:30,17:38
Jan-Apr 2023,8:26,17:27


In [7]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
                    subplot_titles=["Morning peak centre (5:00-10:59)", "Evening peak centre (15:00-19:59)"])

am_minutes = []
pm_minutes = []
am_labels = []
pm_labels = []
for _, row in centres.iterrows():
    am_minutes.append(row["am_centre"] * 60)
    pm_minutes.append(row["pm_centre"] * 60)
    am_labels.append(clock(row["am_centre"]))
    pm_labels.append(clock(row["pm_centre"]))

fig.add_trace(go.Scatter(
    x=centres["month"], y=am_minutes, mode="lines+markers", name="AM",
    line=dict(color=viz.ACCENT, width=2), marker=dict(size=6),
    customdata=am_labels, hovertemplate="%{x|%b %Y}: %{customdata}<extra></extra>",
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=centres["month"], y=pm_minutes, mode="lines+markers", name="PM",
    line=dict(color=viz.CONTRAST, width=2), marker=dict(size=6),
    customdata=pm_labels, hovertemplate="%{x|%b %Y}: %{customdata}<extra></extra>",
), row=2, col=1)


def minute_ticks(first_hour, last_hour):
    values = []
    texts = []
    minute = first_hour * 60
    while minute <= last_hour * 60:
        values.append(minute)
        texts.append(clock(minute / 60))
        minute += 10
    return values, texts


am_values, am_texts = minute_ticks(7, 9)
pm_values, pm_texts = minute_ticks(16, 18)
fig.update_yaxes(tickvals=am_values, ticktext=am_texts, row=1, col=1)
fig.update_yaxes(tickvals=pm_values, ticktext=pm_texts, row=2, col=1)
fig.update_layout(showlegend=False, hovermode="closest", margin=dict(t=130))
fig.add_vrect(x0="2020-03-09", x1="2020-05-31", fillcolor=viz.CONTEXT, opacity=0.3, line_width=0)

title = "Both peaks jumped earlier in 2020 and have been drifting back ever since"
subtitle = "Trip-weighted centre of each peak by month, normal workdays, Jan 2019 to Apr 2023. Grey band = COVID break"
viz.titled(fig, title, subtitle)
viz.save(fig, "05_peak_centre", height=620)
fig

## 4. สถานี × ชั่วโมง

recovery % ของ entries แต่ละสถานีในแต่ละชั่วโมง (วันทำงาน) เรียงสถานีตาม corridor
ไม่รวม MLPT และ BERY (ไม่มีปี 2019) และใช้ชั่วโมง 5-23

In [8]:
station_hour_query = f"""
    CREATE OR REPLACE TEMP TABLE station_hour_recovery AS
    WITH hours_in_period AS (
        SELECT
            h.station::VARCHAR        AS station,
            {period_case("h.date")} AS period,
            h.hour,
            h.entries
        FROM station_hour AS h
        JOIN calendar AS c ON c.date = h.date
        WHERE c.day_status = 'normal' AND c.is_workday AND h.hour >= 5
    ),
    averages AS (
        SELECT station, period, hour, avg(entries) AS entries
        FROM hours_in_period
        WHERE period IS NOT NULL
        GROUP BY station, period, hour
    )
    SELECT
        b.station,
        b.hour,
        b.entries AS entries_2019,
        a.entries AS entries_2023,
        100 * a.entries / nullif(b.entries, 0) AS recovery_pct
    FROM averages AS b
    JOIN averages AS a ON a.station = b.station AND a.hour = b.hour AND a.period = '{AFTER}'
    WHERE b.period = '{BEFORE}'
"""

with stage("05.station_hour_recovery"):
    con.execute(station_hour_query)

recovery_rows = con.sql("SELECT station, hour, recovery_pct, entries_2019, entries_2023 FROM station_hour_recovery").fetchall()

values = {}
for station, hour, recovery_pct, entries_2019, entries_2023 in recovery_rows:
    values[(station, hour)] = (recovery_pct, entries_2019, entries_2023)

station_order = []
for station in STATIONS:
    if (station, 8) in values:
        station_order.append(station)

hours = list(range(5, 24))
matrix = []
hover = []
for station in station_order:
    matrix_row = []
    hover_row = []
    for hour in hours:
        recovery_pct, entries_2019, entries_2023 = values[(station, hour)]
        matrix_row.append(recovery_pct)
        hover_row.append(f"{entries_2023:,.0f} vs {entries_2019:,.0f} entries")
    matrix.append(matrix_row)
    hover.append(hover_row)

print(len(station_order), "stations x", len(hours), "hours")

[perf] 05.station_hour_recovery: 0.01s | 6.99 cores | peak RAM 0.265 GB
48 stations x 19 hours


In [9]:
system_recovery = fetch_value(con, """
    SELECT 100 * sum(entries_2023) / sum(entries_2019) FROM station_hour_recovery
""")
print(f"system recovery of workday entries, hours 5-23: {system_recovery:.1f}%")

heatmap = go.Heatmap(
    z=matrix,
    x=hours,
    y=station_order,
    customdata=hover,
    colorscale=viz.DIVERGING,
    zmid=system_recovery,
    zmin=0,
    zmax=2 * system_recovery,
    colorbar=dict(title="% of 2019", ticksuffix="%"),
    hovertemplate="%{y} %{x}:00 - %{z:.0f}% of 2019<br>%{customdata}<extra></extra>",
)

fig = go.Figure(heatmap)
fig.update_xaxes(title_text="hour", dtick=1)
fig.update_yaxes(autorange="reversed", tickfont=dict(size=10))
fig.update_layout(hovermode="closest")

title = "The weakest cells are both ends of the office commute: suburbs at 8-10, downtowns at 17-20"
subtitle = (
    f"Entries per workday hour, Jan-Apr 2023 as % of Jan-Apr 2019, stations by corridor. "
    f"Grey = system average ({system_recovery:.0f}%), red = below, blue = above"
)
viz.titled(fig, title, subtitle)
viz.save(fig, "05_station_hour_recovery", height=1000)
fig

system recovery of workday entries, hours 5-23: 36.5%


ช่วงที่ฟื้นน้อยที่สุดและมากที่สุด (เฉพาะช่องที่ปี 2019 มีอย่างน้อย 200 คนต่อชั่วโมง เพื่อไม่ให้ช่องเล็กๆ แกว่ง)

In [10]:
extremes_query = """
    SELECT station, hour, round(entries_2019) AS entries_2019, round(entries_2023) AS entries_2023,
           round(recovery_pct, 1) AS recovery_pct
    FROM station_hour_recovery
    WHERE entries_2019 >= 200
    ORDER BY recovery_pct
"""
extremes = con.sql(extremes_query).df()

print("lowest 10:")
print(extremes.head(10).to_string(index=False))
print()
print("highest 10:")
print(extremes.tail(10).to_string(index=False))

lowest 10:
station  hour  entries_2019  entries_2023  recovery_pct
   COLS    23         591.0          54.0           9.2
   WARM     9         891.0          82.0           9.2
   COLS    22         641.0          90.0          14.0
   WARM     8         774.0         111.0          14.4
   WARM    10         314.0          48.0          15.2
   FRMT     9        1570.0         265.0          16.9
   MLBR     9        1179.0         200.0          17.0
   COLM     9         811.0         153.0          18.8
   SBRN     9         632.0         119.0          18.9
   FRMT    10         667.0         129.0          19.3

highest 10:
station  hour  entries_2019  entries_2023  recovery_pct
   SFIA    12         351.0         220.0          62.5
   SFIA    10         285.0         178.0          62.5
   SFIA    20         254.0         158.0          62.5
   HAYW    16         202.0         126.0          62.5
   SFIA    11         331.0         207.0          62.6
   SFIA    14         37

## 5. สัดส่วน peak ราย corridor

สัดส่วนผู้โดยสาร (entries + exits) ที่อยู่ในช่วง peak (6:00-9:59 และ 15:00-18:59) ของวันทำงาน
และ recovery % ของช่วง peak เทียบกับนอก peak

In [11]:
corridor_peak_query = f"""
    WITH hours_in_period AS (
        SELECT
            s.corridor,
            {period_case("h.date")} AS period,
            (h.hour BETWEEN 6 AND 9) OR (h.hour BETWEEN 15 AND 18) AS is_peak,
            h.entries + h.exits AS riders
        FROM station_hour AS h
        JOIN stations AS s ON s.station = h.station::VARCHAR
        JOIN calendar AS c ON c.date = h.date
        WHERE c.day_status = 'normal'
          AND c.is_workday
          AND s.station NOT IN ('MLPT', 'BERY')
    ),
    totals AS (
        SELECT
            corridor,
            period,
            sum(riders) FILTER (WHERE is_peak)     AS peak_riders,
            sum(riders) FILTER (WHERE NOT is_peak) AS off_peak_riders,
            count(*) AS station_hours
        FROM hours_in_period
        WHERE period IS NOT NULL
        GROUP BY corridor, period
    ),
    workdays AS (
        SELECT {period_case("date")} AS period, count(*) AS days
        FROM calendar
        WHERE day_status = 'normal' AND is_workday
        GROUP BY period
    )
    SELECT
        b.corridor,
        round(100 * b.peak_riders / (b.peak_riders + b.off_peak_riders), 1) AS peak_share_2019,
        round(100 * a.peak_riders / (a.peak_riders + a.off_peak_riders), 1) AS peak_share_2023,
        round(100 * (a.peak_riders / wa.days) / (b.peak_riders / wb.days), 1)         AS peak_recovery_pct,
        round(100 * (a.off_peak_riders / wa.days) / (b.off_peak_riders / wb.days), 1) AS off_peak_recovery_pct
    FROM totals AS b
    JOIN totals AS a ON a.corridor = b.corridor AND a.period = '{AFTER}'
    JOIN workdays AS wb ON wb.period = '{BEFORE}'
    JOIN workdays AS wa ON wa.period = '{AFTER}'
    WHERE b.period = '{BEFORE}'
    ORDER BY peak_recovery_pct
"""

corridor_peak = con.sql(corridor_peak_query).df()
corridor_peak

,corridor,peak_share_2019,peak_share_2023,peak_recovery_pct,off_peak_recovery_pct
0,Dublin line,71.4,68.6,31.0,35.5
1,SF Downtown,69.5,67.5,31.6,34.7
2,Oakland Core,68.3,65.6,35.8,40.3
3,Concord-Antioch line,71.7,68.8,35.8,41.1
4,Fremont line,69.7,67.5,37.1,41.1
5,Peninsula,66.9,61.8,37.6,47.0
6,SF Mission/South,66.8,64.9,38.6,41.9
7,Berkeley-Richmond line,65.9,63.7,41.3,45.4


In [12]:
segment_x = []
segment_y = []
for _, row in corridor_peak.iterrows():
    segment_x.extend([row["off_peak_recovery_pct"], row["peak_recovery_pct"], None])
    segment_y.extend([row["corridor"], row["corridor"], None])

segments = go.Scatter(x=segment_x, y=segment_y, mode="lines", line=dict(color=viz.GRID, width=4),
                      showlegend=False, hoverinfo="skip")
peak_points = go.Scatter(
    x=corridor_peak["peak_recovery_pct"], y=corridor_peak["corridor"], mode="markers", name="peak hours",
    marker=dict(color=viz.CONTRAST, size=12),
    hovertemplate="%{y}: peak hours %{x:.1f}% of 2019<extra></extra>",
)
off_peak_points = go.Scatter(
    x=corridor_peak["off_peak_recovery_pct"], y=corridor_peak["corridor"], mode="markers", name="off-peak hours",
    marker=dict(color=viz.ACCENT, size=12),
    hovertemplate="%{y}: off-peak hours %{x:.1f}% of 2019<extra></extra>",
)

fig = go.Figure([segments, peak_points, off_peak_points])
fig.update_layout(hovermode="closest", margin=dict(l=200))
fig.update_xaxes(title_text="workday riders, % of Jan-Apr 2019", ticksuffix="%", showgrid=True, gridcolor=viz.GRID)
fig.update_yaxes(showgrid=False)

title = "In every corridor, peak hours recovered less than off-peak hours"
subtitle = "Workday riders (entries + exits) in Jan-Apr 2023 as % of Jan-Apr 2019, peak (6-9, 15-18) vs off-peak"
viz.titled(fig, title, subtitle)
viz.save(fig, "05_corridor_peak_recovery")
fig

## สรุป: Insight → Pain point

pain point ของ BART ในข้อนี้คือ ตารางเดินรถออกแบบไว้สำหรับ peak ก่อน COVID ทำให้รถอาจเกินในบางชั่วโมงและแน่นในบางชั่วโมง

| # | Insight (ตัวเลข) | Pain point | ตอบโจทย์แค่ไหน | ข้อเสนอ |
|---|---|---|---|---|
| 1 | ชั่วโมงที่ฟื้นน้อยที่สุดคือ 9:00 (30%) และ 18:00 (31%) ส่วนกลางวัน 11:00-14:00 ฟื้นได้ 46-50% | รถเกินช่วง peak | ตรงจุดเรื่องความต้องการรายชั่วโมง แต่บอกจำนวนขบวนที่เกินไม่ได้ เพราะไม่มีข้อมูลความจุ | ลดความยาวขบวนช่วง 8:00-9:59 และ 18:00-19:59 ก่อนช่วงอื่น และไม่ลดบริการกลางวันในสัดส่วนเดียวกัน |
| 2 | peak แบนลง ชั่วโมงที่คนเยอะที่สุดเหลือ 11.8% ของวัน (เดิม 12.4%) และต้องใช้ 6 ชั่วโมงถึงจะได้ครึ่งหนึ่งของทริป (เดิม 5) ชั่วโมงที่คนเยอะที่สุดเปลี่ยนจาก 8:00 เป็น 17:00 | ตารางเดินรถที่ออกแบบให้ peak เช้าแหลม | ตรงจุด | กระจายความถี่ให้สม่ำเสมอขึ้นตลอดวัน ลดความต่างระหว่าง peak กับนอก peak |
| 3 | peak ทั้งสองเลื่อนเร็วขึ้นทันทีในปี 2020 (เช้า ~8:04, เย็น ~17:16) แล้วค่อยๆ เลื่อนกลับ ปี 2023 อยู่ที่ 8:26 และ 17:27 (ปี 2019 คือ 8:30 และ 17:38) | เวลาเริ่ม-จบของบริการช่วง peak | ตรงจุด | ไม่ต้องเลื่อนตารางทั้งชุด แต่ peak เย็นยังเร็วกว่าเดิม ~10 นาที ควรเริ่มเพิ่มรถช่วงเย็นเร็วขึ้นเล็กน้อย และติดตามต่อเพราะยังเลื่อนกลับอยู่ |
| 4 | ทุก corridor ช่วง peak ฟื้นน้อยกว่านอก peak ห่างมากที่สุดที่ Peninsula (38% กับ 47%) | ขนาดขบวนช่วง peak ราย corridor | ตรงจุด | ลดขนาดขบวน peak ของสายที่ peak หายมาก (Peninsula, Concord-Antioch, Oakland Core) ก่อน |
| 5 | ช่องที่ฟื้นน้อยที่สุดคือทั้งสองฝั่งของการเดินทางไปทำงาน: สถานีชานเมืองช่วง 8:00-10:59 (เช่น WARM, FRMT, MLBR, COLM, NCON) และ downtown SF/Oakland ช่วง 17:00-20:59 | ช่องว่างอยู่ที่ไหน | ตรงจุด | ใช้ตาราง สถานี × ชั่วโมง นี้เลือกขบวนที่ลดได้ เช่น ขบวนเช้าที่ออกจากปลายสายชานเมือง |
| 6 | COLS ช่วง 22:00-23:59 เหลือแค่ 9-14% ของปี 2019 | บริการรอบดึก | บางส่วน ตรงกับที่ Warriors ย้ายไป SF ปลายปี 2019 และ Raiders ย้ายไป Las Vegas ปี 2020 แต่ข้อมูลชุดนี้ยืนยันสาเหตุไม่ได้ | ทบทวนบริการเสริมหลังเลิกงานกีฬาที่ Coliseum |

ข้อจำกัด
- ไม่มีตารางเดินรถและความจุขบวน จึงบอกได้แค่ว่าความต้องการเปลี่ยนไปอย่างไร
- exits ใช้ชั่วโมงเดียวกับเวลาเข้า
- ปี 2023 มีข้อมูลแค่ ม.ค.-เม.ย.

ต่อไป: 06_q3_load_flows